# NB-22 — the reanalysis, rerun with the 100-hour replicate

**CPU only. No GPU. No model is loaded. Nothing is trained or attacked.**
Set the accelerator to *None* before running. This notebook spends zero GPU quota.

## Why this notebook exists

`analysis/reanalysis.py` ran on 2026-10-29 and answered the supervisor review.
At that point every exposure above 24.6 h came from one training run, so
dropping that run put the slope's interval across zero: a fact about the
design, not the effect.

NB-19 trained a second 100-hour run (seed 1338) and NB-21 attacked all three of
its milestones. This notebook reruns the same script with that run included
and prints the old and new numbers side by side.

What the replicate does: it gives the top of the exposure range a second,
independent cluster. What it does NOT do: break the collinearity between
exposure and training step, because both 100-hour runs were checkpointed at the
same steps.

**Already done locally on 2026-11-04** (CSVs from the GitHub repo plus the three
NB-21 downloads). Result: 12 clusters, beta1 +0.0888, CR1 [+0.024, +0.154],
no single-run drop includes zero, wild cluster bootstrap p = 0.0608. This
notebook is only needed to reproduce that on Kaggle.

## What this notebook also settles

1. **Row counts.** Section 5.1 of the paper quotes the 100-hour *final*
   checkpoint at 46% on 80 of 100 utterances. `tab:pairs` quotes *ms-2000* at
   46% on 100. Same number, different conditions. This prints the row count and
   sha256 of every file so the coincidence is documented rather than assumed.
2. **The rung-dependent sigma, tested.** The 10.4x ratio has so far been a
   ratio of two point estimates. This adds an F test and an exact permutation
   test, so it becomes a result with a p-value.

## Inputs to attach

The full list, with owners, is in `notebooks/NB-22_INPUTS.md`. In short:
`reanalysis.py` as a one-file dataset, seven `eps50-*` datasets, the NB-12
output, the four NB-16 notebook outputs, and the NB-21 outputs. 17 CSVs in all.

Internet **OFF**. Accelerator **None**.


## Step 0 — getting `reanalysis.py` onto Kaggle

One-time, about a minute. On your machine:

```
Kaggle -> Datasets -> New Dataset
  title: thesis-analysis-code
  upload: analysis/reanalysis.py
```

Then attach it here. The notebook asserts the file is present and refuses to
guess — a missing input must fail loudly, not silently produce a smaller fit.

If you would rather not upload anything: download the 16 CSVs into
`data/results/` on your own machine and run

```
python3 analysis/reanalysis.py .
```

It is pure standard library and needs no numpy, no pandas and no GPU. This
notebook exists for the case where the CSVs are easier to reach on Kaggle than
locally.


In [ ]:
import glob, hashlib, io, json, os, shutil, subprocess, sys, math, itertools, csv

print("python", sys.version.split()[0])
print("cwd   ", os.getcwd())

WORK = "/kaggle/working"
RES  = os.path.join(WORK, "data", "results")
os.makedirs(RES, exist_ok=True)

# --- locate reanalysis.py -------------------------------------------------
SCRIPT = None
for d in range(1, 5):
    hits = sorted(glob.glob("/kaggle/input/" + "*/" * d + "reanalysis.py"))
    if hits:
        SCRIPT = hits[0]
        break
if SCRIPT is None and os.path.exists(os.path.join(WORK, "reanalysis.py")):
    SCRIPT = os.path.join(WORK, "reanalysis.py")

assert SCRIPT, (
    "reanalysis.py not found in any attached input.\n"
    "Upload analysis/reanalysis.py as a Kaggle dataset and attach it.\n"
    "See step 0. Refusing to continue: a missing script would otherwise be\n"
    "silently replaced by whatever this notebook reimplemented, which is the\n"
    "NB-06 bug class."
)
LOCAL = os.path.join(WORK, "reanalysis.py")
if os.path.abspath(SCRIPT) != os.path.abspath(LOCAL):
    shutil.copy(SCRIPT, LOCAL)
print("script:", SCRIPT)
print("sha256:", hashlib.sha256(open(LOCAL, "rb").read()).hexdigest()[:16])

src = open(LOCAL).read()
assert "REPLICATE_100H" in src, (
    "the attached reanalysis.py predates the 100-hour replicate patch.\n"
    "Upload the patched version (it defines REPLICATE_100H and RUNG100_PAIRS)."
)
print("patched version confirmed: REPLICATE_100H present")


## Step 1 — stage the CSVs, and audit every one of them

Explicit filename list. No regex over condition names. A pattern that silently
fails to match is how NB-06 produced a false result and how NB-17 currently
drops all three new conditions.

Two attachments can carry the same filename, and the cell handles each case
explicitly rather than taking whichever path sorts first:

| case | example | treatment |
|---|---|---|
| identical copies | the same dataset mounted twice | use it |
| **resume chain** | NB-16 Run A's 80-row `rung1-s1339` and Run D's 100-row one; NB-21 session A's 80-row `ms2000` and session C's 100-row one | take the larger, print it |
| **conflict** | two copies that disagree on any shared utterance | **stop** |

One file is special. The 61% re-attack of the 1-hour trajectory weights (Task B
run 7) is published in `eps50-rung1-traj-final-t5-n100` under the **same
basename as the real 1-hour final**, `eps50_rung1_final_T5_n100.csv`. It is
staged as `eps50_rung1-traj-SECOND-MEASUREMENT_T5_n100.csv`, taken only from
that dataset, and that dataset is excluded when staging the real 1-hour final.

Tested locally against a fake input tree before handover: both resume chains
resolve to the 100-row copy, the 1-hour final reads 75% and the re-attack 61%,
a disagreeing duplicate stops the run, and a missing re-attack dataset stops it
with a message naming where the file comes from.

Every file must end at exactly 100 rows.


In [ ]:
INPUT_ROOT = globals().get("INPUT_ROOT", "/kaggle/input")

FILES = [
    # --- the published ladder, 13 conditions -----------------------------
    "eps50_rung1_final_T5_n100.csv",
    "eps50_rung1-trajfinal_T5_n100.csv",
    "eps50_rung1-s1338_T5_n100.csv",
    "eps50_rung1-s1339_T5_n100.csv",
    "eps50_rung5_final_T5_n100.csv",
    "eps50_rung5-s1338_T5_n100.csv",
    "eps50_rung5-s1339_T5_n100.csv",
    "eps50_rung25_final_T5_n100.csv",
    "eps50_rung25-s1338_T5_n100.csv",
    "eps50_rung25-s1339_T5_n100.csv",
    "eps50_rung100_milestone-1000_T5_n100.csv",
    "eps50_rung100_milestone-2000_T5_n100.csv",
    "eps50_rung100_milestone-3500_T5_n100.csv",
    # --- the re-attack of identical weights (attack-measurement variance) ---
    "eps50_rung1-traj-SECOND-MEASUREMENT_T5_n100.csv",
    # --- the 100-hour replicate, seed 1338, NB-21 -------------------------
    "eps50_rung100-s1338-ms1000_T5_n100.csv",
    "eps50_rung100-s1338-ms2000_T5_n100.csv",
    "eps50_rung100-s1338-ms3500_T5_n100.csv",
]

# The 61% re-attack (Task B run 7) is published in eps50-rung1-traj-final-t5-n100
# under the SAME basename as the real 1-hour final (75%). It is staged under the
# repository's distinct name, taken ONLY from that dataset, and that dataset is
# excluded when staging the real 1-hour final. Without this, whichever path
# sorts first silently wins.
TRAJ_RUN7 = "eps50-rung1-traj-final-t5-n100"
SPECIAL = {"eps50_rung1-traj-SECOND-MEASUREMENT_T5_n100.csv":
           ("eps50_rung1_final_T5_n100.csv", TRAJ_RUN7)}
EXCLUDE = {"eps50_rung1_final_T5_n100.csv": [TRAJ_RUN7]}

def find_all(basename):
    hits = []
    for d in range(1, 5):
        hits += sorted(glob.glob(INPUT_ROOT + "/" + "*/" * d + basename))
    return list(dict.fromkeys(hits))

def read_rows(path):
    with io.open(path, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def outcome(r):
    c = str(r["censored"]).strip().lower() in ("true", "1", "yes")
    e = r.get("eps50", "").strip()
    return (c, None if c or not e else float(e))

def same(o1, o2):
    if o1[0] != o2[0]:
        return False
    if o1[1] is None or o2[1] is None:
        return o1[1] == o2[1]
    return abs(o1[1] - o2[1]) <= 1e-9 * max(abs(o1[1]), abs(o2[1]))

def asr(rows):
    return 100.0 * sum(1 for r in rows if not outcome(r)[0]) / max(len(rows), 1)

def resolve(fn, paths):
    """Return (chosen_path, note). Identical copies are fine. A resume chain
    (every smaller copy a strict subset of the largest, agreeing on every
    shared utt_id) takes the largest. Anything else is a conflict: stop."""
    by_sha = {}
    for p in paths:
        by_sha.setdefault(hashlib.sha256(open(p, "rb").read()).hexdigest(), []).append(p)
    if len(by_sha) == 1:
        return paths[0], "identical x%d" % len(paths)
    cands = []
    for sha, ps in by_sha.items():
        rows = read_rows(ps[0])
        cands.append((len(rows), ps[0], {r["utt_id"]: outcome(r) for r in rows}, rows))
    cands.sort(key=lambda c: -c[0])
    n_big, p_big, big, rows_big = cands[0]
    for n, p, m, rows in cands[1:]:
        extra = set(m) - set(big)
        bad = [u for u in m if u in big and not same(m[u], big[u])]
        if extra or bad or n == n_big:
            print("\n*** CONFLICT on %s ***" % fn)
            print("   %3d rows  ASR %5.1f%%  %s" % (n_big, asr(rows_big), p_big))
            print("   %3d rows  ASR %5.1f%%  %s" % (n, asr(rows), p))
            if bad:
                print("   they disagree on %d shared utterance(s), e.g. %s" % (len(bad), bad[0]))
            if extra:
                print("   the smaller copy has %d utterance(s) the larger lacks" % len(extra))
            raise AssertionError("conflicting copies of %s -- detach one" % fn)
    notes = ", ".join("%d-row subset ignored" % c[0] for c in cands[1:])
    return p_big, "resume chain: %d rows used, %s" % (n_big, notes)

missing, audit = [], []
for fn in FILES:
    if fn in SPECIAL:
        src_name, slug = SPECIAL[fn]
        paths = [p for p in find_all(src_name) if "/" + slug + "/" in p]
    else:
        paths = [p for p in find_all(fn)
                 if not any("/" + s + "/" in p for s in EXCLUDE.get(fn, []))]
    if not paths:
        missing.append(fn)
        continue
    chosen, note = resolve(fn, paths)
    shutil.copy(chosen, os.path.join(RES, fn))
    rows = read_rows(chosen)
    audit.append(dict(file=fn, rows=len(rows), asr=asr(rows),
                      sha=hashlib.sha256(open(chosen, "rb").read()).hexdigest()[:12],
                      note=note, src=chosen))

print("%-48s %5s %7s  %-12s %s" % ("file", "rows", "ASR", "sha256[:12]", "note"))
print("-" * 100)
for a in audit:
    flag = "" if a["rows"] == 100 else "   <== NOT 100 ROWS"
    print("%-48s %5d %6.1f%%  %-12s %s%s" % (a["file"], a["rows"], a["asr"], a["sha"], a["note"], flag))
print("\nsources:")
for a in audit:
    print("  %-48s <- %s" % (a["file"], a["src"]))

if missing:
    print("\n*** MISSING, attach the input that carries each: ***")
    for m in missing:
        hint = (" (comes from %s, filed there as %s)" % (SPECIAL[m][1], SPECIAL[m][0])
                if m in SPECIAL else "")
        print("   ", m + hint)

assert not missing, "%d input files not found -- see the list above" % len(missing)
bad = [a for a in audit if a["rows"] != 100]
assert not bad, "these files do not have 100 rows: %s" % [b["file"] for b in bad]
print("\nAll %d files present, all 100 rows." % len(audit))


### What the row-count audit settles

If `eps50_rung100_milestone-2000_T5_n100.csv` has 100 rows, then the paper's
`tab:pairs` entry for pair 2 is a full 100-utterance measurement and the 46%
it shares with the 100-hour *final* checkpoint (80 of 100 utterances, §5.1) is
a coincidence, not a transcription error.

The ANCOVA already implies this — 13 conditions and 1,300 observations means
100 rows each — but implying is not the same as printing it.


## Step 2 — the rung-dependent sigma, with a test attached

So far the 10.4x figure is a ratio of two point estimates. A reviewer is
entitled to ask whether three pairs that happen to be close could arise by
chance from the same variance as the four that are far apart.

Two tests, because they fail in different ways:

* **F test on the variance ratio.** Efficient, and sensitive to non-normality
  at df 4 and 3 — which is exactly the regime here, so it is reported as
  indicative.
* **Exact permutation test.** All $\binom{7}{3}=35$ ways of splitting the seven
  observed pair-differences into a group of four and a group of three. Makes no
  distributional assumption. Note the floor: with this design the smallest
  attainable one-sided p is $1/35 = 0.0286$, so a reviewer should read it as
  "as extreme as the design permits", not as a strong p-value.

Pairing is by **optimiser step**, never by rung. The three milestones of one
trajectory are three exposures, not three replicates.


In [ ]:
def c4(df):
    return math.exp(0.5 * math.log(2.0 / df) + math.lgamma((df + 1) / 2.0)
                    - math.lgamma(df / 2.0))

def pooled_var_from_pairs(diffs):
    """Each pair of runs gives variance diff^2/2 on 1 df."""
    v = [d * d / 2.0 for d in diffs]
    return sum(v) / len(v), len(v)

def betacf(a, b, x, itmax=300, eps=3e-16):
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c, d = 1.0, 1.0 - qab * x / qap
    if abs(d) < 1e-300: d = 1e-300
    d = 1.0 / d
    h = d
    for m in range(1, itmax + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-300: d = 1e-300
        c = 1.0 + aa / c
        if abs(c) < 1e-300: c = 1e-300
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-300: d = 1e-300
        c = 1.0 + aa / c
        if abs(c) < 1e-300: c = 1e-300
        d = 1.0 / d
        de = d * c
        h *= de
        if abs(de - 1.0) < eps:
            break
    return h

def betainc(a, b, x):
    """Regularized incomplete beta I_x(a,b)."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    lbeta = (math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
             + a * math.log(x) + b * math.log(1.0 - x))
    if x < (a + 1.0) / (a + b + 2.0):
        return math.exp(lbeta) * betacf(a, b, x) / a
    return 1.0 - math.exp(lbeta) * betacf(b, a, 1.0 - x) / b

def f_sf(f, d1, d2):
    """P(F_{d1,d2} > f)."""
    if f <= 0: return 1.0
    return betainc(d2 / 2.0, d1 / 2.0, d2 / (d2 + d1 * f))

# --- self-test against values that can be checked by hand ----------------
assert abs(c4(1) - 0.7978845608) < 1e-9
assert abs(c4(2) - 0.8862269255) < 1e-9
assert abs(c4(3) - 0.9213177319) < 1e-9
assert abs(c4(4) - 0.9399856030) < 1e-7
assert abs(betainc(2.0, 3.0, 0.5) - 0.6875) < 1e-12   # closed form
assert abs(f_sf(1.0, 4, 4) - 0.5) < 1e-12             # median of F(d,d)
assert abs(f_sf(9.1172, 4, 3) - 0.05) < 2e-4          # F table, alpha=0.05
print("numerics self-test passed")


In [ ]:
# ---------------------------------------------------------------------------
# The seven replicate pairs. Low-rung diffs come from the published sigma
# groups; high-rung diffs come from the NB-21 milestone pairs.
# ---------------------------------------------------------------------------
LOW = [("1 h, tf4.44.2", 41.0, 75.0),
       ("1 h, tf5.16.1", 39.0, 57.0),
       ("5 h, tf5.16.1", 59.0, 96.0),
       ("25 h, tf5.16.1", 57.0, 78.0)]
HIGH = [("step 1000, 38.80 h", 37.0, 34.0),
        ("step 2000, 77.70 h", 46.0, 48.0),
        ("step 3500, 98.02 h", 53.0, 56.0)]

print("  %-24s %8s %8s %8s" % ("pair", "run A", "run B", "|diff|"))
for lab, a, b in LOW + HIGH:
    print("  %-24s %7.1f%% %7.1f%% %7.1f pp" % (lab, a, b, abs(b - a)))

dlow  = [abs(b - a) for _, a, b in LOW]
dhigh = [abs(b - a) for _, a, b in HIGH]

vlow,  dflow  = pooled_var_from_pairs(dlow)
vhigh, dfhigh = pooled_var_from_pairs(dhigh)
slow,  shigh  = math.sqrt(vlow), math.sqrt(vhigh)
clow,  chigh  = slow / c4(dflow), shigh / c4(dfhigh)

print("\n  %-34s %8s %4s %9s" % ("group", "raw sd", "df", "sigma"))
print("  %-34s %7.3f %4d %8.3f" % ("1, 5, 25 h  (4 pairs)", slow, dflow, clow))
print("  %-34s %7.3f %4d %8.3f" % ("100 h       (3 pairs)", shigh, dfhigh, chigh))
print("\n  ratio of c4-corrected sigmas: %.2fx" % (clow / chigh))

F = vlow / vhigh
p_f = f_sf(F, dflow, dfhigh)
print("\n--- F test on the variance ratio -----------------------------------")
print("  F(%d,%d) = %.2f      one-sided p = %.5f" % (dflow, dfhigh, F, p_f))
print("  Indicative only: the F test assumes normal pair differences and")
print("  df 4 against df 3 is where that assumption bites hardest.")

alld = dlow + dhigh
n_hi = len(dhigh)
obs = F
extreme = 0
ratios = []
for combo in itertools.combinations(range(len(alld)), n_hi):
    hi = [alld[i] for i in combo]
    lo = [alld[i] for i in range(len(alld)) if i not in combo]
    vh, _ = pooled_var_from_pairs(hi)
    vl, _ = pooled_var_from_pairs(lo)
    r = vl / vh if vh > 0 else float("inf")
    ratios.append(r)
    if r >= obs - 1e-12:
        extreme += 1
p_perm = extreme / len(ratios)
print("\n--- exact permutation test -----------------------------------------")
print("  all C(7,3) = %d splits of the seven observed pair-differences" % len(ratios))
print("  observed variance ratio %.2f is rank %d of %d (largest = 1)"
      % (obs, sorted(ratios, reverse=True).index(max(r for r in ratios if r >= obs - 1e-12)) + 1, len(ratios)))
print("  exact one-sided p = %d/%d = %.4f" % (extreme, len(ratios), p_perm))
print("  Floor: 1/%d = %.4f is the smallest p this design can produce."
      % (len(ratios), 1.0 / len(ratios)))

print("\n--- sign of the three high-rung gaps -------------------------------")
signed = [round(b - a, 1) for _, a, b in HIGH]
print("  s1338 minus s1337: %s" % signed)
print("  Not a constant offset between the two runs, so it reads as noise")
print("  rather than a systematic difference between seeds or libraries.")

RUNG_SIGMA = dict(sigma_low=clow, df_low=dflow, sigma_high=chigh, df_high=dfhigh,
                  ratio=clow / chigh, F=F, p_F=p_f, p_perm=p_perm,
                  diffs_low=dlow, diffs_high=dhigh, signed_high=signed)


## Step 3 — rerun the reanalysis both ways

Same script, same CSVs, same estimator. The only difference is whether the
three `100h s1338` conditions are in `COND`.

`INCLUDE_100H_REPLICATE=0` reproduces the 2026-10-29 output exactly, so the
comparison is against a number that is already on record rather than against a
memory of one.


In [ ]:
def run(include):
    env = dict(os.environ)
    env["INCLUDE_100H_REPLICATE"] = "1" if include else "0"
    r = subprocess.run([sys.executable, LOCAL, WORK], capture_output=True,
                       text=True, env=env)
    if r.returncode != 0:
        print(r.stdout[-4000:])
        print(r.stderr[-4000:])
        raise RuntimeError("reanalysis.py exited %d" % r.returncode)
    return r.stdout

out_old = run(False)
out_new = run(True)
open(os.path.join(WORK, "reanalysis-output-published.txt"), "w").write(out_old)
open(os.path.join(WORK, "reanalysis-output-with-replicate.txt"), "w").write(out_new)
print("published  : %6d chars" % len(out_old))
print("+ replicate: %6d chars" % len(out_new))


In [ ]:
import re

def grab(txt, pat, cast=float, all_=False):
    m = re.findall(pat, txt)
    if not m:
        return None
    return [cast(x) for x in m] if all_ else cast(m[0])

def headline(txt):
    d = {}
    d["n"] = grab(txt, r"n = (\d+) utterance observations", int)
    d["G"] = grab(txt, r"G = (\d+) training-run clusters", int)
    d["beta1"] = grab(txt, r"beta1 = ([+-][\d.]+) per decade")
    ci = re.search(r"95% CI \[([+-][\d.]+), ([+-][\d.]+)\]\s+->", txt)
    d["ci"] = (float(ci.group(1)), float(ci.group(2))) if ci else None
    d["boot_p"] = grab(txt, r"bootstrap p = ([\d.]+)")
    m = re.search(r"drop 100h-r1\s+beta1\s+([+-][\d.]+)\s+CI \[([+-][\d.]+), ([+-][\d.]+)\]\s+(\S+)", txt)
    d["loco_100h_r1"] = (float(m.group(1)), float(m.group(2)), float(m.group(3)), m.group(4)) if m else None
    m = re.search(r"drop 100h-s1338\s+beta1\s+([+-][\d.]+)\s+CI \[([+-][\d.]+), ([+-][\d.]+)\]\s+(\S+)", txt)
    d["loco_100h_s1338"] = (float(m.group(1)), float(m.group(2)), float(m.group(3)), m.group(4)) if m else None
    d["one_cluster"] = "Every exposure above 24.6 h sits in ONE cluster" in txt
    n_incl = len(re.findall(r"INCLUDES 0", txt))
    d["n_includes_zero"] = n_incl
    return d

A, B = headline(out_old), headline(out_new)

print("=" * 78)
print("  HEADLINE NUMBERS, published fit vs fit with the 100-hour replicate")
print("=" * 78)
rows = [
    ("utterance observations",      A["n"],     B["n"]),
    ("training-run clusters G",     A["G"],     B["G"]),
    ("beta1 per decade",            A["beta1"], B["beta1"]),
    ("CR1 95% CI",                  A["ci"],    B["ci"]),
    ("wild cluster bootstrap p",    A["boot_p"],B["boot_p"]),
    ("exposure >24.6h in 1 cluster",A["one_cluster"], B["one_cluster"]),
    ("'INCLUDES 0' verdicts",       A["n_includes_zero"], B["n_includes_zero"]),
]
for lab, a, b in rows:
    print("  %-32s %-24s %-24s" % (lab, a, b))

print("\n--- the diagnostic this replicate was run for ----------------------")
print("  leave-one-cluster-out, dropping the ORIGINAL 100-hour run:")
print("    published  : %s" % (A["loco_100h_r1"],))
print("    + replicate: %s" % (B["loco_100h_r1"],))
print("  leave-one-cluster-out, dropping the REPLICATE:")
print("    + replicate: %s" % (B["loco_100h_s1338"],))
print("""
  Read it this way. In the published fit, dropping 100h-r1 removed every
  exposure above 24.6 hours at once, so the interval had to widen across zero.
  That is a statement about the DESIGN, not about the effect. With a second
  100-hour run present, each drop leaves the other one in, so the diagnostic
  finally tests what it is supposed to test.""")

if B["boot_p"] is not None and A["boot_p"] is not None:
    print("\n  bootstrap p moved %.4f -> %.4f" % (A["boot_p"], B["boot_p"]))
    if B["boot_p"] >= 0.05:
        print("  Still not significant at 5%. Report it. Do not re-pool to chase it.")
    else:
        print("  Now below 5%. Say plainly that this became significant only after")
        print("  the replicate was added, and that the pre-registered fit did not.")


In [ ]:
summary = dict(
    notebook="NB-22-reanalysis-with-100h-replicate",
    rung_sigma=RUNG_SIGMA,
    published=A, with_replicate=B,
    audit=[{k: a[k] for k in ("file", "rows", "sha")} for a in audit],
)
with open(os.path.join(WORK, "nb22_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)

lines = [
    "### NB-22 - reanalysis with the 100-hour replicate",
    "",
    "| Paper value | Source |",
    "|---|---|",
    "| rung 1/5/25 sigma: %.3f pp, df %d | 4 pairs, c4-corrected |" % (RUNG_SIGMA["sigma_low"], RUNG_SIGMA["df_low"]),
    "| rung 100 sigma: %.3f pp, df %d | 3 milestone pairs, c4-corrected |" % (RUNG_SIGMA["sigma_high"], RUNG_SIGMA["df_high"]),
    "| ratio: %.2fx | the two rows above |" % RUNG_SIGMA["ratio"],
    "| F(%d,%d) = %.2f, one-sided p = %.5f | variance-ratio test, indicative |" % (
        RUNG_SIGMA["df_low"], RUNG_SIGMA["df_high"], RUNG_SIGMA["F"], RUNG_SIGMA["p_F"]),
    "| exact permutation p = %.4f | all C(7,3)=35 splits; floor 1/35 = 0.0286 |" % RUNG_SIGMA["p_perm"],
    "| signed high-rung gaps: %s | no constant offset between runs |" % (RUNG_SIGMA["signed_high"],),
    "| beta1 %s -> %s | published fit -> with replicate |" % (A["beta1"], B["beta1"]),
    "| G %s -> %s, n %s -> %s | cluster and observation counts |" % (A["G"], B["G"], A["n"], B["n"]),
    "| wild bootstrap p %s -> %s | same estimator, same weights, same seed |" % (A["boot_p"], B["boot_p"]),
    "| LOCO drop 100h-r1: %s -> %s | the diagnostic the replicate was run for |" % (
        A["loco_100h_r1"][3] if A["loco_100h_r1"] else "?",
        B["loco_100h_r1"][3] if B["loco_100h_r1"] else "?"),
    "",
    "All 17 input CSVs verified at 100 rows; sha256 prefixes in `nb22_summary.json`.",
]
open(os.path.join(WORK, "nb22_for_NUMBERS.md"), "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
print("\nwrote:")
for f in sorted(os.listdir(WORK)):
    if f.startswith("nb22") or f.startswith("reanalysis-output"):
        print("   ", f, os.path.getsize(os.path.join(WORK, f)), "B")
print("""
PUBLISH NOTE. This notebook's working directory also holds data/results/, which
is a copy of the attached inputs. Publishing the whole output as a dataset would
republish 17 CSVs you already have published. Download the four small files
above instead, or publish under a NEW name -- never over an eps50-* dataset.""")


## What to do with the output

1. Paste `nb22_for_NUMBERS.md` into `paper/NUMBERS.md`.
2. Replace `analysis/reanalysis-output.txt` with
   `reanalysis-output-with-replicate.txt`, and keep the published one beside it
   as `reanalysis-output-published.txt`. Both are then on record.
3. The leave-one-cluster-out paragraph in the Limitations section says the
   77.7-hour level is single-run and that the diagnostic rests on one cluster at
   the top. Rewrite it from the new output.

**Do not** quietly swap the headline fit for whichever version has the smaller
p-value. If the bootstrap only clears 5% with the replicate included, that is
worth saying in those words — a result that depends on an addition made after
the review is a weaker result than one that did not, and a reader who finds that
out for themselves will discount everything else.
